# RAG Amélioré — Pipeline Contextual Retrieval + Hybrid Search + Re-ranking

**Améliorations par rapport à la v1 :**
1. **Contextual Retrieval** : chaque chunk porte son contexte (chapitre + concept + type) — réduit les erreurs de retrieval de ~49% (Anthropic Research)
2. **Chunking typé** : chaque formule, exemple, exercice est tagué → filtrage précis selon l'intention
3. **Intent detection** : détecte si la question porte sur une formule, un exemple, un exercice…
4. **Cross-encoder re-ranking** : reclasse les résultats avec un modèle de pertinence croisée (plus précis que la similarité cosinus seule)
5. **Meilleur modèle d'embedding** : `multilingual-mpnet-base-v2`

In [ ]:
!pip install -q chromadb sentence-transformers rank-bm25
# Cross-encoder (reranking) — nécessite torch
!pip install -q transformers torch --index-url https://download.pytorch.org/whl/cpu

## Bloc 1 — Imports & Configuration

In [ ]:
import json
import re
import sys
import collections
from pathlib import Path

# ── Détection automatique de la racine du projet ─────────────────────────────
# Fonctionne que le notebook soit lancé depuis notebooks/ ou depuis la racine.
def _trouver_racine(marqueur="config.yaml"):
    p = Path.cwd()
    for _ in range(5):
        if (p / marqueur).exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = _trouver_racine()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── Configuration ────────────────────────────────────────────────────────────

# JSON structuré généré par structuration_json.py — modifier le nom si besoin
DATASET_PATH = str(PROJECT_ROOT / "data" / "pdfs" / "Statistique_cours.json")

EMBEDDING_MODEL    = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
CROSS_ENCODER_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

TOP_CHILDREN = 15
TOP_RERANKED = 5
TOP_PARENTS  = 3
RRF_K        = 60

print(f'Racine projet : {PROJECT_ROOT}')
print(f'Dataset path  : {DATASET_PATH}')
print('Configuration chargee')

## Bloc 2 — Chargement du dataset structuré

In [ ]:
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

titre = dataset.get('titre_annale', 'Cours')
meta  = dataset.get('metadata', {})
parties = dataset.get('parties', [])

print(f'Dataset : {titre}')
print(f'Domaine : {meta.get("domaine")} | Niveau : {meta.get("niveau")}')
print(f'Nombre de parties : {len(parties)}')
for p in parties:
    print(f'  - {p.get("titre")} ({len(p.get("concepts", []))} concepts)')

## Bloc 3 — Chunking Contextuel Hiérarchique

Chaque chunk atomique (formule, exemple, exercice...) porte un **préfixe contextuel** indiquant sa position dans le cours. C'est la méthode "Contextual Retrieval" publiée par Anthropic qui réduit les erreurs de retrieval de ~49% par rapport aux chunks sans contexte.

```
[Chapitre: X | Concept: Y | Type: formule]
Formule de la variance : Var(X) = ...
```

In [ ]:
from chunking.chunker import construire_chunks, Chunk

# construire_chunks(dataset) → list[Chunk]
# Chaque Chunk a : .texte, .chunk_id, .chapitre, .concept,
#                  .type_concept, .type_chunk, .parent_content, .source
# Chunk.as_dict() → dict compatible ChromaDB (valeurs scalaires)

chunks    = construire_chunks(dataset, source=Path(DATASET_PATH).stem)
textes    = [c.texte for c in chunks]
metadatas = [c.as_dict() for c in chunks]
ids       = [c.chunk_id for c in chunks]

# Statistiques
from collections import Counter
type_counts = Counter(c.type_chunk for c in chunks)
print(f'{len(chunks)} chunks construits depuis {Path(DATASET_PATH).name}')
print('Repartition par type :')
for t, c in type_counts.most_common():
    print(f'  {t:12s}: {c}')

## Bloc 4 — Indexation : Embeddings + ChromaDB + BM25

In [ ]:
# ── Modèle d'embedding ────────────────────────────────────────────────────────
print(f'Chargement du modele : {EMBEDDING_MODEL}...')
embed_model = SentenceTransformer(EMBEDDING_MODEL)

# ── ChromaDB ─────────────────────────────────────────────────────────────────
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection('cours_rag_v2')
    print('Ancienne collection supprimee')
except:
    print('Nouvelle collection')

# cosine = meilleure distance pour les embeddings normalisés
collection = chroma_client.get_or_create_collection(
    name='cours_rag_v2',
    metadata={'hnsw:space': 'cosine'}
)

print(f'Vectorisation de {len(textes)} chunks...')
embeddings = embed_model.encode(
    textes,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # important pour la similarité cosinus
).tolist()

collection.add(
    embeddings=embeddings,
    documents=textes,
    metadatas=metadatas,
    ids=ids
)
print(f'ChromaDB : {collection.count()} chunks indexes')

# ── BM25 (recherche lexicale) ─────────────────────────────────────────────────
def tokenizer(texte):
    if not texte:
        return []
    texte = texte.lower().replace("'", ' ').replace('-', ' ')
    texte = re.sub(r'[^\w\s]', ' ', texte)
    return [w for w in texte.split() if len(w) > 1]

corpus_tokenise = [tokenizer(t) for t in textes]
bm25 = BM25Okapi(corpus_tokenise)
print('BM25 indexe')

## Bloc 5 — Cross-encoder (Re-ranking)

In [ ]:
# Le cross-encoder évalue la pertinence CONJOINTE (question, passage)
# au lieu de comparer des vecteurs séparément → bien plus précis
# mais plus lent, donc on l'applique seulement sur les TOP_CHILDREN candidats

RERANKING_DISPONIBLE = False
try:
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    RERANKING_DISPONIBLE = True
    print(f'Cross-encoder charge : {CROSS_ENCODER_MODEL}')
except Exception as e:
    print(f'Cross-encoder non disponible ({e}) — on utilisera RRF seul')

## Bloc 6 — Détection d'intention + Recherche hybride RRF

In [ ]:
# ── Détection d'intention ─────────────────────────────────────────────────────
# Permet de filtrer ChromaDB sur le type de chunk le plus pertinent
# sans sacrifier la recherche générale (les deux sont fusionnées via RRF)

INTENTIONS = {
    'formule':    ['formule', 'equation', 'expression', 'calcul', 'calculer',
                   'calculez', 'definir', 'calculons', 'formulas', 'comment calculer'],
    'exemple':    ['exemple', 'application', 'illustrer', 'illustration', 'montrer',
                   'appliquer', 'concret'],
    'exercice':   ['exercice', 'td', 'travaux', 'probleme', 'corrig', 'entrainement',
                   'solution', 'resoudre'],
    'preuve':     ['demonstration', 'preuve', 'demontrer', 'montrer que', 'justifier',
                   'pourquoi', 'comment obtenir'],
    'definition': ['definition', 'definir', 'qu est ce que', 'c est quoi', 'signifie',
                   'veut dire', 'expliquer'],
}

def detecter_intention(query):
    q = query.lower()
    scores = {
        intent: sum(1 for kw in mots if kw in q)
        for intent, mots in INTENTIONS.items()
    }
    meilleur = max(scores, key=scores.get)
    return meilleur if scores[meilleur] > 0 else None


# ── Recherche hybride RRF ─────────────────────────────────────────────────────

def recherche_hybride(query, top_k=TOP_CHILDREN):
    """
    Combine recherche vectorielle (ChromaDB) et lexicale (BM25) via RRF.
    Si une intention est détectée, la recherche filtrée par type reçoit
    un poids doublé dans la fusion.
    """
    intention = detecter_intention(query)
    rrf_scores = collections.defaultdict(float)

    # Vectoriser la requête
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    # A. Recherche vectorielle générale (poids 1)
    res_vec = collection.query(query_embeddings=[q_vec], n_results=top_k)
    for rang, cid in enumerate(res_vec['ids'][0]):
        rrf_scores[cid] += 1.0 / (RRF_K + rang + 1)

    # B. Recherche vectorielle filtrée par intention (poids 2 si pertinente)
    if intention:
        try:
            res_filtre = collection.query(
                query_embeddings=[q_vec],
                n_results=top_k,
                where={'type_chunk': intention}
            )
            for rang, cid in enumerate(res_filtre['ids'][0]):
                rrf_scores[cid] += 2.0 / (RRF_K + rang + 1)
        except Exception:
            pass

    # C. BM25 lexicale (poids 1)
    tokens = tokenizer(query)
    scores_bm25 = bm25.get_scores(tokens)
    top_bm25 = sorted(range(len(scores_bm25)), key=lambda i: scores_bm25[i], reverse=True)[:top_k]
    for rang, idx in enumerate(top_bm25):
        rrf_scores[ids[idx]] += 1.0 / (RRF_K + rang + 1)

    candidats = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return candidats[:top_k], intention


# ── Re-ranking croisé ─────────────────────────────────────────────────────────

def reranker(query, candidats_ids, top_n=TOP_RERANKED):
    """
    Reclasse les chunks candidats en évaluant la pertinence (query, passage)
    conjointement via le cross-encoder. Plus précis que la similarité cosinus.
    """
    if not RERANKING_DISPONIBLE or not candidats_ids:
        return candidats_ids[:top_n]

    res = collection.get(ids=candidats_ids[:20])
    textes_map = {cid: doc for cid, doc in zip(res['ids'], res['documents'])}

    paires = [(query, textes_map[cid]) for cid in candidats_ids[:20] if cid in textes_map]
    if not paires:
        return candidats_ids[:top_n]

    scores = cross_encoder.predict(paires)
    tries = sorted(
        zip(candidats_ids[:20], scores),
        key=lambda x: x[1],
        reverse=True
    )
    return [cid for cid, _ in tries[:top_n]]


# ── Extraction des contextes parents ─────────────────────────────────────────

def extraire_contextes_parents(ids_tries, max_parents=TOP_PARENTS):
    """
    Pour chaque chunk sélectionné, remonte au contexte parent complet.
    Déduplique les parents pour ne pas envoyer deux fois le même concept au LLM.
    """
    vus = set()
    contextes = []

    for cid in ids_tries:
        res = collection.get(ids=[cid])
        if not res['metadatas']:
            continue
        meta = res['metadatas'][0]
        parent = meta.get('parent_content', '')

        if parent not in vus:
            vus.add(parent)
            contextes.append({
                'concept':        meta.get('concept', ''),
                'chapitre':       meta.get('chapitre', ''),
                'type_chunk':     meta.get('type_chunk', ''),
                'parent_content': parent,
            })

        if len(contextes) >= max_parents:
            break

    return contextes


print('Moteur de recherche hybride pret')

## Bloc 7 — Pipeline RAG complet

In [ ]:
import google.generativeai as genai

# Remplace par ta clé API Gemini
GEMINI_API_KEY = 'VOTRE_CLE_GEMINI_ICI'
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-2.5-flash')


def pipeline_rag(query, verbose=True):
    """
    Pipeline RAG complet :
      1. Détection d'intention
      2. Recherche hybride (vecteurs + BM25 + RRF)
      3. Re-ranking croisé
      4. Remontée des contextes parents
      5. Génération de réponse (LLM)
    """
    if verbose:
        print(f'\nQuestion : {query}')
        print('-' * 70)

    # Étape 1 — Intention
    candidats, intention = recherche_hybride(query, top_k=TOP_CHILDREN)
    if verbose:
        print(f'Intention detectee  : {intention or "generale"}')
        print(f'Candidats trouves   : {len(candidats)}')

    # Étape 2 — Re-ranking
    ids_rerankes = reranker(query, candidats, top_n=TOP_RERANKED)
    if verbose and RERANKING_DISPONIBLE:
        print(f'Apres re-ranking    : {len(ids_rerankes)} chunks retenus')

    # Étape 3 — Contextes parents
    contextes = extraire_contextes_parents(ids_rerankes, max_parents=TOP_PARENTS)
    if verbose:
        print(f'Contextes parents   : {len(contextes)}')
        for i, ctx in enumerate(contextes):
            print(f'  {i+1}. [{ctx["type_chunk"]}] {ctx["concept"]} — {ctx["chapitre"]}')

    # Étape 4 — Construction du contexte pour le LLM
    contexte_llm = ''
    for i, ctx in enumerate(contextes):
        contexte_llm += (
            f'\n\n=== Contexte {i+1} : {ctx["concept"]} '
            f'(Chapitre : {ctx["chapitre"]}) ===\n'
            f'{ctx["parent_content"]}'
        )

    # Étape 5 — Prompt structuré
    prompt = f"""Tu es un professeur expert en "{titre}" pour des etudiants universitaires.
Reponds a la question de l'etudiant de facon claire, rigoureuse et pedagogique,
en te basant UNIQUEMENT sur le contenu du cours fourni ci-dessous.
Si une information n'est pas dans le contexte, dis-le clairement au lieu d'inventer.

[CONTENU DU COURS]
{contexte_llm}

[QUESTION DE L'ETUDIANT]
{query}

[CONSIGNES]
- Utilise les formules exactes du cours (en LaTeX entre $$ si possible)
- Illustre avec les exemples du cours quand ils sont pertinents
- Structure avec des titres clairs (##) et des listes
- Sois precis et complet mais concis
- N'invente pas d'informations absentes du contexte

Reponse du Professeur :"""

    # Étape 6 — Génération
    if verbose:
        print('\nGeneration de la reponse...')
        print('=' * 70)

    try:
        rep = gemini.generate_content(prompt)
        print(rep.text)
    except Exception as e:
        print(f'LLM indisponible ({e}).\n\nContexte recupere :\n{contexte_llm}')


print('Pipeline RAG pret')

## Bloc 8 — Tests

In [ ]:
# Test 1 — question sur une formule
pipeline_rag('Comment calculer la variance et l ecart-type ?')

In [ ]:
# Test 2 — question sur un exercice
pipeline_rag('Montrez la solution de l exercice du poste de péage')

In [ ]:
# Test 3 — définition
pipeline_rag("Qu'est-ce que l'intervalle de confiance ?")

In [ ]:
# Test 4 — preuve
pipeline_rag("Demonstration de l'esperance de la moyenne empirique")

## Bloc 9 — Diagnostic des chunks récupérés (debug)

Utile pour vérifier que le bon contenu est bien récupéré pour une question donnée.

In [ ]:
def diagnostiquer(query):
    """Affiche les chunks candidats et leur score RRF avant génération."""
    candidats, intention = recherche_hybride(query, top_k=TOP_CHILDREN)
    ids_rerankes = reranker(query, candidats, top_n=TOP_RERANKED)

    print(f'Question  : {query}')
    print(f'Intention : {intention or "generale"}')
    print(f'\nTop {len(ids_rerankes)} chunks apres re-ranking :')
    print('-' * 70)

    for i, cid in enumerate(ids_rerankes):
        res = collection.get(ids=[cid])
        if res['documents']:
            meta = res['metadatas'][0]
            texte = res['documents'][0]
            print(f'\n{i+1}. [{meta["type_chunk"]}] {meta["concept"]} ({meta["chapitre"]})')
            print(f'   {texte[:200]}...')


diagnostiquer('Quelle est la formule de la médiane ?')